## Programming Fundamentals

Gratuito Databricks: https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/814341352205843?o=7474657872577658#command/8635188974282960

## D. MapReduce in Action

MapReduce is the classic pattern for parallel data processing, and it is exactly what a DataFrame pipeline compiles down to under the hood. 

Consider this small ETL job, which reads sales records, filters them, and totals the amount per ID:

In [0]:

# MapReduce in Action

# MapReduce is the classic pattern for parallel data processing, and it is exactly what a DataFrame pipeline compiles down to. Consider this small ETL job, which reads sales records, filters them, and totals the amount per id:
    
# READ: scan the source file into a DataFrame
#input_df = spark.read.csv("sales_data.csv")

# MAP: narrow, per-partition transformations - no data movement
#filtered_df = input_df.select("id", "amt").filter("amt > 50")

# SHUFFLE: groupBy sets the key, so all rows for the same id must be
# moved onto the same partition (the exchange) before aggregation
#grouped_df = filtered_df.groupBy("id")

# REDUCE: aggregate each group after the shuffle - sum amt per id
#aggregated_df = grouped_df.agg({"amt": "sum"})

# WRITE: each partition writes its slice of the result
#aggregated_df.write.csv("/Volumes/catalog/schema/output/sales_by_id")



Explicação Rápida das Etapas

READ (Leitura):
Lê os dados brutos da fonte (sales_data.csv) para dentro de um DataFrame do Spark.

MAP (Mapeamento / Transformações Estreitas):
Executa operações de seleção (select) e filtragem (filter) de forma independente em cada partição, sem mover dados entre os nós do cluster (operações narrow).

SHUFFLE (Embaralhamento / Troca):
O comando groupBy("id") define a chave de agrupamento. Para que o cálculo funcione, todas as linhas com o mesmo id que estão espalhadas pelo cluster precisam ser movidas através da rede para a mesma partição (etapa de exchange).

REDUCE (Redução / Agregação):
Após o shuffle, cada grupo é consolidado e agregado em paralelo (neste caso, somando o campo amt por cada id).

WRITE (Escrita):
Cada nó/partição do cluster grava a sua respetiva fração do resultado final no destino especificado.

In [0]:
%python
# READ: create a small sales dataset (inline so the demo is self-contained)
input_df = spark.createDataFrame(
    [(1, 30), (1, 80), (2, 60), (2, 20), (3, 90), (3, 55)],
    ["id", "amt"]
)
display(input_df)
# READ: scan the source file into a DataFrame
#input_df = spark.read.csv("sales_data.csv")

# MAP: narrow, per-partition transformations - no data movement
filtered_df = input_df.select("id", "amt").filter("amt > 50")

# SHUFFLE: groupBy sets the key, so all rows for the same id must be
# moved onto the same partition (the exchange) before aggregation
grouped_df = filtered_df.groupBy("id")

# REDUCE: aggregate each group after the shuffle - sum amt per id
aggregated_df = grouped_df.agg({"amt": "sum"})

# WRITE: display the result (in practice each partition writes its slice)
display(aggregated_df)

# WRITE: each partition writes its slice of the result
#aggregated_df.write.csv("/Volumes/catalog/schema/output/sales_by_id")

id,amt
1,30
1,80
2,60
2,20
3,90
3,55


id,sum(amt)
1,80
2,60
3,145



## Handling Missing Values
Real data has gaps. Common functions for handling nulls:
    • isNull() / isNotNull() - check if values are null
    • count(col) - counts non-null values in a specific column
    • df.fillna() / df.na.fill() - replace nulls with values
    • df.dropna() / df.na.drop() - remove rows with nulls
# Drop rows with nulls in a specific column
df.dropna(subset=["my_column"])

# ..or..
df.na.drop(subset=["my_column"])

# Replace nulls with a value
df.fillna("unknown")

# ..or..
df.na.fill("unknown")


## B. Handling Missing Values

Real-world data always contains gaps and missing values (`null`s). PySpark provides several built-in functions and methods to detect, clean, or replace them:

* **`isNull()` / `isNotNull()`:** Conditional functions to check if values are null.
* **`count(col)`:** Counts only non-null values in a specific column.
* **`df.fillna()` / `df.na.fill()`:** Replaces null values with specified substitutes (constants, maps, or column defaults).
* **`df.dropna()` / `df.na.drop()`:** Removes rows containing null values based on specific thresholds or subsets.

In [0]:
%python
import pyspark.sql.functions as F


df = spark.createDataFrame(
    [(1, 30), (1, 80), (2, 60), (2, 20), (3, 90), (3, 55), ( 4,None), (5,None)],
    ["id", "amt"]
)
display(df)



# 1. Dropping rows with nulls in a specific column
df_cleaned = df.dropna(subset=["amt"])
display(df_cleaned)
# Alternatively, using the .na namespace:
# df_cleaned = df.na.drop(subset=["amt"])

# 2. Replacing nulls with a default value (e.g., "unknown" for strings)
df_filled = df.fillna("unknown")
display(df_filled)
# Alternatively, using the .na namespace:
# df_filled = df.na.fill("unknown")

# 3. Checking for nulls and counting non-null entries
df.select(
    F.count("amt").alias("non_null_count"),
    F.sum(F.col("amt").isNull().cast("int")).alias("null_count")
).display()

id,amt
1,30
1,80
2,60
2,20
3,90
3,55
4,null
5,null


id,amt
1,30
1,80
2,60
2,20
3,90
3,55


id,amt
1,30
1,80
2,60
2,20
3,90
3,55
4,null
5,null


non_null_count,null_count
6,2


In [0]:
import pyspark.sql.functions as F

# fillna("unknown") didn't fill nulls above because "amt" is a long column.
# Spark only applies a string fill value to string-type columns.
# Cast "amt" to string first, then fillna works:
df_str = df.withColumn("amt", F.col("amt").cast("string"))
df_filled = df_str.fillna("unknown")
display(df_filled)

id,amt
1,30
1,80
2,60
2,20
3,90
3,55
4,unknown
5,unknown



# User Defined Functions

UDFs allow you to apply your own Python functions to DataFrame columns:




In [0]:
from pyspark.sql.functions import udf

# define your function
#@udf("string")
#def make_greeting(name):
#    return f"Hello, {name}!"

# use your function
#df.select(make_greeting("name"))


    • UDFs allow developers to create reusable custom functions
    
    • UDFs can adversely impact performance: they cannot be optimized by the Catalyst optimizer and add serialization overhead between the JVM and the Python worker

## F. User Defined Functions (UDFs)

UDFs allow you to apply custom Python functions to DataFrame columns:

* UDFs allow developers to create reusable custom functions.
* **Performance Warning:** UDFs can adversely impact performance because they cannot be optimized by the Catalyst optimizer and add serialization overhead between the JVM and the Python worker. Whenever possible, prefer built-in Spark SQL functions.

In [0]:
%python
from pyspark.sql.functions import udf

# Define your custom function using the @udf decorator
@udf("string")
def make_greeting(name):
    return f"Hello, {name}!"

# Use your custom UDF function in a DataFrame selection
df_result = df.select(make_greeting("id"))

display(df_result)

make_greeting(id)
"Hello, 1!"
"Hello, 1!"
"Hello, 2!"
"Hello, 2!"
"Hello, 3!"
"Hello, 3!"
"Hello, 4!"
"Hello, 5!"


In [0]:
display(df)

id,amt
1,30
1,80
2,60
2,20
3,90
3,55
4,null
5,null


## Pandas UDFs and Apache Arrow

Pandas UDFs let you write Python functions that operate on batches of rows instead of single rows, leveraging Apache Arrow for more efficient Python-JVM serialization.
from pyspark.sql.functions import pandas_udf


    • More efficient than their non-pandas counterparts

    • In Spark 4 you can also accelerate row-based UDFs with Arrow serialization: @udf("string", useArrow=True)
    
    • You should still use a built-in function if one is available - built-ins are always fastest because the optimizer understands them

In [0]:

from pyspark.sql.functions import pandas_udf

# define your pandas udf
@pandas_udf("integer")
def age_plus_one(age_series):
    return age_series + 1

# use your pandas udf
df.select(age_plus_one("amt"))



DataFrame[age_plus_one(amt): int]

In [0]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- amt: long (nullable = true)



In [0]:
display(df)

id,amt
1,30
1,80
2,60
2,20
3,90
3,55
4,null
5,null


## Pandas UDFs and Apache Arrow

Pandas UDFs let you write Python functions that operate on batches of rows instead of single rows, leveraging **Apache Arrow** for much more efficient Python-JVM serialization.

* **Performance:** Significantly more efficient than standard row-based UDF counterparts.
* **Spark 4 Feature:** In Spark 4, you can also accelerate standard row-based UDFs with Arrow serialization using `@udf("string", useArrow=True)`.
* **Best Practice:** You should still use a built-in function if one is available — built-ins are always the fastest choice because the Catalyst optimizer understands them fully.

In [0]:
%python
from pyspark.sql.functions import pandas_udf

# Define your pandas UDF operating on Pandas Series
@pandas_udf("integer")
def age_plus_one(age_series):
    return age_series + 1

# Use your pandas UDF in a DataFrame operation
df_result = df.select(age_plus_one("amt"))

display(df_result)

age_plus_one(amt)
31
81
61
21
91
56
null
null
